In [1]:
import uproot
import awkward as ak
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import math
import hist
import vector
print("uproot version",uproot.__version__)
print("awkward version",ak.__version__)
print("numpy version",np.__version__)
print("matplotlib version",matplotlib.__version__)
print("hist version",hist.__version__)
print("vector version",vector.__version__)
from scipy.optimize import curve_fit
from scipy.special import erf 

vector.register_awkward() 

import os

uproot version 5.5.1
awkward version 2.7.2
numpy version 2.1.3
matplotlib version 3.9.0
hist version 2.8.0
vector version 1.5.2


In [2]:
def crystal_ball(x, N, mu, sigma, alpha, n):
    t = (x - mu) / sigma
    A = (n / abs(alpha))**n * np.exp(-alpha**2 / 2)
    B = n / abs(alpha) - abs(alpha)
    C = (n / abs(alpha)) / (n - 1) * np.exp(-alpha**2 / 2)
    D = np.sqrt(np.pi / 2) * (1 + erf(abs(alpha) / np.sqrt(2)))
    norm = sigma * (C + D)
    result = np.piecewise(
        t,
        [t > -alpha, t <= -alpha],
        [lambda t: N * np.exp(-t**2 / 2) / norm,
         lambda t: N * A * (B - t)**(-n) / norm]
    )
    return result

In [ ]:
folder_path = "/pbs/throng/training/nantes-m2-rps-exp/data"
results = []
j_psi_gen = []
j_psi_rec = []

file_count =0

for file_name in os.listdir(folder_path):
    if file_name.endswith(".mc.root"):
        file_path = os.path.join(folder_path, file_name)
        results.append(file_name[0:-8])

        print(f"Traitement du fichier : {file_name}")
        file = uproot.open(file_path)

        gen = file["genTree"]
        m = gen.arrays(["nMuonsGen","Muon_GenPx","Muon_GenPy","Muon_GenPz","Muon_GenE","Muon_GenLabel","Muon_GenMotherPDGCode"])
        
        nMuonsGen = 0
        for i in range(len(m)):
            info = m[i]["nMuonsGen"]
            nMuonsGen += info
        print("nMuonsGen = ", nMuonsGen)

        events = file["eventsTree"]
        n = events.arrays(["zVtx","isCMUL","nMuons","Muon_Px","Muon_Py","Muon_Pz","Muon_E","Muon_Charge","Muon_thetaAbs","Muon_matchedTrgThreshold","Muon_MCLabel", "isCMSH", "isCMLL", "Muon_yDCA"])

        nMuons = 0
        for i in range(len(n)):
            info = n[i]["nMuons"]
            nMuons += info
        print("nMuons = ", nMuons)

        print("nMuons/nMuonsGen = ", nMuons/nMuonsGen)
        
        #-----------------------Coupure---------------------------------------------

        mask = (n["nMuons"] >= 2) & ak.all(m["Muon_GenMotherPDGCode"] == 443, axis=1) 
        
        # filtered_gen_events = m[ak.all(m["Muon_GenMotherPDGCode"] == 443, axis=1)]
        # filtered_events = n[mask & (n.isCMUL)&(abs(n.zVtx)<10)]
        # print(len(filtered_gen_events))
        # print(len(filtered_events))
        # j_psi_gen.append(len(filtered_gen_events))
        # j_psi_rec.append(len(filtered_events))

        #--------------------------Figure-------------------------------

        filtered_events1 = n[mask]
        print("filtered_events",len(filtered_events1)) 
        filtered_events = n[mask & (n.isCMUL)&(abs(n.zVtx)<10)]
        
        muon_vectors = ak.zip(
            {
                "px": filtered_events.Muon_Px,
                "py": filtered_events.Muon_Py,
                "pz": filtered_events.Muon_Pz,
                "E": filtered_events.Muon_E,
                "matchedTrgThreshold": filtered_events.Muon_matchedTrgThreshold,
                "DCA": filtered_events.Muon_yDCA,
                "charge": filtered_events.Muon_Charge,
                "theta": filtered_events.Muon_thetaAbs,
            },
            with_name="Momentum4D",
        )
        
        # Calculer les masses invariantes pour chaque paire de muons dans chaque événement
        muon_pairs = ak.combinations(muon_vectors, 2, fields=["muon1", "muon2"])

                #------------coupure-----------

        opposite_charge_pairs = muon_pairs[
            (muon_pairs.muon1.charge != muon_pairs.muon2.charge)
            & (muon_pairs.muon1.matchedTrgThreshold == 2)
            & (muon_pairs.muon2.matchedTrgThreshold == 2)
            & (muon_pairs.muon1.DCA <= 1.5)
            & (muon_pairs.muon2.DCA <= 1.5)
            & (muon_pairs.muon1.theta <= 10)
            & (muon_pairs.muon2.theta <= 10)
            & (muon_pairs.muon1.theta >= 3)
            & (muon_pairs.muon2.theta >= 3)
            & (muon_pairs.muon1.eta <= -2.5)
            & (muon_pairs.muon1.eta <= -2.5)
            & (muon_pairs.muon2.eta >= -4)
            & (muon_pairs.muon2.eta >= -4)
        ]

        masses_opposite = (opposite_charge_pairs.muon1 + opposite_charge_pairs.muon2).mass
        y_pair = (opposite_charge_pairs.muon1 + opposite_charge_pairs.muon2).rapidity
        
        # Appliquer le filtre de rapidité
        filtered_masses = masses_opposite[(y_pair <= -2.5) & (y_pair >= -4)]

                #----
        
        # Créer un histogramme des valeurs de la masse
        flattened_opposite = ak.flatten(filtered_masses[(filtered_masses >= 0) & (filtered_masses <= 10)], axis=None)
        
        # hist, bin_edges = np.histogram(flattened_opposite, bins=500, range=(0, 5))
        # plt.hist(flattened_opposite, bins=500)
            
        # plt.xlabel('Masse invariante (GeV/c^2)')
        # plt.ylabel("Nombre d'événements")
        # plt.xlim(0,5)
        # plt.ylim(0,1200)
        # #plt.yscale('log')
        # plt.title('Distribution des masses invariantes des paires de muons (0-5 GeV/c^2)')
        # plt.grid(True)
        # plt.show()
        
        print("masses_opposit",len(masses_opposite)) #nb j/psi
        



        #--------------------------------Fit-----------------------------

        bin_centers = np.asarray((bin_edges[:-1] + bin_edges[1:]) / 2)
        y_data = np.asarray(hist)
        valid_indices = (bin_centers >= 2.1) & (bin_centers <= 3.4)
        x_peak = bin_centers[valid_indices]
        y_peak = y_data[valid_indices]
        
        
        try:
            popt_cb2, pcov_cb2 = curve_fit(
                crystal_ball, x_peak, y_peak,
                p0=[np.max(y_peak), 3.0, 0.1, 1.5, 5],
                bounds=([0, 2.1, 0.01, 0.1, 1], [np.inf, 3.4, 1, 5, 10]),
                maxfev=200000
            )
        except RuntimeError as e:
            print("Erreur d'ajustement Crystal Ball :", e)
        
        # Générer les courbes ajustées
        x_fit = np.linspace(2.1, 3.4, 1000)
        y_cb2 = crystal_ball(x_fit, *popt_cb2) if 'popt_cb2' in locals() else None
        
        # Résultats de l'ajustement
        #if 'popt_cb2' in locals():
            #print(f"Paramètres CB2 : {popt_cb2}")
        
        # Tracer les données et les ajustements
        # plt.step(bin_centers, y_data, where='mid', label='Histogramme des données', color='blue', alpha=0.7)
        # if y_cb2 is not None:
        #     plt.plot(x_fit, y_cb2, label='Crystal Ball (CB2)', color='red')
        
        # # Mise en forme
        # plt.xlabel('Masse invariante (GeV/c^2)')
        # plt.ylabel('Nombre d\'événements')
        # plt.title('Ajustement des fonctions de signal (CB2)')
        # plt.legend()
        # plt.grid(True)
        # plt.show()

      

        #-------------------------------------Pour n fichier seulement---------a supprimer plus tard (supprimer l'affichage surtout)--------------------
        
        # file_count += 1
        # if file_count >= 10 :
        #     print("Limite de fichiers atteinte.")
        #     break

Traitement du fichier : run292140.mc.root
nMuonsGen =  14000
nMuons =  7742
nMuons/nMuonsGen =  0.553
filtered_events 1235
masses_opposit 943
Traitement du fichier : run290721.mc.root
nMuonsGen =  40000
nMuons =  21341
nMuons/nMuonsGen =  0.533525
filtered_events 3301
masses_opposit 2546
Traitement du fichier : run292160.mc.root
nMuonsGen =  260000
nMuons =  143259
nMuons/nMuonsGen =  0.5509961538461539
filtered_events 23104
masses_opposit 17549
Traitement du fichier : run291976.mc.root
nMuonsGen =  140000
nMuons =  77005
nMuons/nMuonsGen =  0.5500357142857143
filtered_events 12287
masses_opposit 9322
Traitement du fichier : run291944.mc.root
nMuonsGen =  100000
nMuons =  55365
nMuons/nMuonsGen =  0.55365
filtered_events 8901
masses_opposit 6860
Traitement du fichier : run291942.mc.root
nMuonsGen =  60000
nMuons =  33146
nMuons/nMuonsGen =  0.5524333333333333
filtered_events 5337
masses_opposit 4129
Traitement du fichier : run292062.mc.root
nMuonsGen =  240000
nMuons =  131802
nMuons/n

In [4]:
import os
import uproot
import awkward as ak
import numpy as np

folder_path = "/pbs/throng/training/nantes-m2-rps-exp/data"
valid_files = []   # Liste des fichiers corrects
corrupt_files = [] # Liste des fichiers corrompus

total_nMuonsGen = 0
total_nMuons = 0
total_files = 0

for file_name in os.listdir(folder_path):
    if file_name.endswith(".mc.root"):
        file_path = os.path.join(folder_path, file_name)

        try:
            # Tenter d'ouvrir le fichier
            file = uproot.open(file_path)

            # Vérifier la présence des arbres
            if "genTree" not in file or "eventsTree" not in file:
                print(f"Fichier corrompu : {file_name} (arbres manquants)")
                corrupt_files.append(file_name)
                continue

            # Charger les données des arbres
            gen = file["genTree"]
            events = file["eventsTree"]

            try:
                m = gen.arrays(["nMuonsGen", "Muon_GenPx", "Muon_GenPy", "Muon_GenPz",
                                "Muon_GenE", "Muon_GenLabel", "Muon_GenMotherPDGCode"])
                n = events.arrays(["zVtx", "isCMUL", "nMuons", "Muon_Px", "Muon_Py", "Muon_Pz",
                                   "Muon_E", "Muon_Charge", "Muon_thetaAbs",
                                   "Muon_matchedTrgThreshold", "Muon_MCLabel", 
                                   "isCMSH", "isCMLL", "Muon_yDCA"])
            except Exception as e:
                print(f"Fichier corrompu : {file_name} (erreur lors de la lecture des branches : {e})")
                corrupt_files.append(file_name)
                continue

            # Vérification du contenu des branches
            if len(m) == 0 or len(n) == 0:
                print(f"Fichier corrompu : {file_name} (données vides)")
                corrupt_files.append(file_name)
                continue

            # Calculer le nombre total de muons générés et reconstruits
            nMuonsGen = ak.sum(m["nMuonsGen"])
            nMuons = ak.sum(n["nMuons"])

            # Vérifier que les valeurs sont cohérentes
            if nMuonsGen == 0 or nMuons == 0 or np.isnan(nMuonsGen) or np.isnan(nMuons):
                print(f"Fichier corrompu : {file_name} (valeurs incohérentes)")
                corrupt_files.append(file_name)
                continue

            # Si tout est bon, ajouter aux listes et afficher les résultats
            valid_files.append(file_name)
            total_nMuonsGen += nMuonsGen
            total_nMuons += nMuons
            total_files += 1

            print(f"Traitement du fichier : {file_name} | nMuons = {nMuons} | nMuonsGen = {nMuonsGen} | ratio = {nMuons/nMuonsGen}")

        except Exception as e:
            print(f"Fichier corrompu : {file_name} (erreur générale : {e})")
            corrupt_files.append(file_name)
            continue

# Vérification finale des valeurs globales
if total_files == 0 or total_nMuonsGen == 0 or total_nMuons == 0:
    print("\nAucun fichier valide trouvé ou problème global détecté !")
else:
    print("\nTous les fichiers semblent valides.")



Traitement du fichier : run292140.mc.root | nMuons = 7742 | nMuonsGen = 14000 | ratio = 0.553
Traitement du fichier : run290721.mc.root | nMuons = 21341 | nMuonsGen = 40000 | ratio = 0.533525
Traitement du fichier : run292160.mc.root | nMuons = 143259 | nMuonsGen = 260000 | ratio = 0.5509961538461539
Traitement du fichier : run291976.mc.root | nMuons = 77005 | nMuonsGen = 140000 | ratio = 0.5500357142857143
Traitement du fichier : run291944.mc.root | nMuons = 55365 | nMuonsGen = 100000 | ratio = 0.55365
Traitement du fichier : run291942.mc.root | nMuons = 33146 | nMuonsGen = 60000 | ratio = 0.5524333333333333
Traitement du fichier : run292062.mc.root | nMuons = 131802 | nMuonsGen = 240000 | ratio = 0.549175
Traitement du fichier : run291692.mc.root | nMuons = 44194 | nMuonsGen = 80000 | ratio = 0.552425
Traitement du fichier : run291283.mc.root | nMuons = 46241 | nMuonsGen = 80000 | ratio = 0.5780125
Traitement du fichier : run292164.mc.root | nMuons = 8771 | nMuonsGen = 16000 | ratio 